# RNN으로_금융앱리뷰_분석하기

In [1]:
# !pip install konlpy

In [6]:
from konlpy.tag import Mecab
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Embedding
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [7]:
train_data=pd.read_csv("../05machine_learning/data/bank_app_reviews_train.csv")
train_data.head()

,리뷰일,평점,사용자리뷰,업체답변,은행명
0,2023-12-21,5,엄빠 폰에 설치해드렸는데 두분 다 쓰기 편하다고 하시네요 ㅎㅎ 저도 쓰고있음!,NaN,하나
1,2025-02-17,1,Cd기 축소 연장하려면 이 어플 깔라는데 왜 30퍼에서 안깔리는지 아니 애초에 슈퍼...,안녕하세요. 신한은행입니다.\n먼저 SOL사용에 불편을 드려 죄송합니다.\n어플의 ...,신한
2,2024-07-26,5,서비스가 통합되어 있어서 점점 사용빈도가 높아지네요.,고객님 안녕하세요! KB스타뱅킹을 이용해 주셔서 진심으로 감사드립니다. KB스타뱅킹...,국민
3,2024-09-01,1,카드 충천이 안됌,"안녕하세요. 전민구 님, 토스팀입니다. 남겨주신 내용만으로는 겪고계신 불편사항의 자...",토스
4,2023-11-13,1,알뜰폰 인증이 안돼요.....,"안녕하세요 헤이모두들안녕님, 우리은행입니다. 먼저 이용에 불편을 드려 매우 죄송합니...",우리


특수문자 제거 함수

In [8]:
import re

def clean_text(text):
    cleaned=re.sub(r'[^가-힣a-zA-Z0-9\s]','',text) #한글, 영문, 숫자
    cleaned=re.sub(r'\s+',' ', cleaned) # 연속된 공백을 하나의 공백
    return cleaned.strip()


사용자 리뷰에서 특수문자 제거

In [9]:
train_data['사용자리뷰']=train_data['사용자리뷰'].apply(clean_text)

is_good 컬럼 추가, 평점 4이상은 긍정: 1, 3이하는 부정:0

In [10]:
train_data['is_good']=train_data['평점'].apply(lambda x: 1 if x>=4 else 0)
train_data['is_good']

0        1
1        0
2        1
3        0
4        0
        ..
22241    0
22242    0
22243    1
22244    1
22245    0
Name: is_good, Length: 22246, dtype: int64

토큰화

In [11]:
mecab=Mecab()
mecab.morphs(train_data['사용자리뷰'][0])

['엄',
 '빠',
 '폰',
 '에',
 '설치',
 '해',
 '드렸',
 '는데',
 '두',
 '분',
 '다',
 '쓰',
 '기',
 '편하',
 '다고',
 '하',
 '시',
 '네요',
 '저',
 '도',
 '쓰',
 '고',
 '있',
 '음']

전체 문장을 토큰화 한 후 tokenized_docs에 저장

In [12]:
tokenized_docs=train_data['사용자리뷰'].apply(mecab.morphs)

In [13]:
tokenized_docs[0]

['엄',
 '빠',
 '폰',
 '에',
 '설치',
 '해',
 '드렸',
 '는데',
 '두',
 '분',
 '다',
 '쓰',
 '기',
 '편하',
 '다고',
 '하',
 '시',
 '네요',
 '저',
 '도',
 '쓰',
 '고',
 '있',
 '음']

In [14]:
tokenized_docs[1]

['Cd',
 '기',
 '축소',
 '연장',
 '하',
 '려면',
 '이',
 '어',
 '플',
 '깔',
 '라는데',
 '왜',
 '30',
 '퍼',
 '에서',
 '안',
 '깔리',
 '는지',
 '아니',
 '애초',
 '에',
 '슈퍼',
 '쏠',
 '나왔',
 '으면',
 '이',
 '어',
 '플',
 '없애',
 '고',
 '슈퍼',
 '솔',
 '에',
 '기능',
 '넣',
 '어',
 '주',
 '던가',
 '왜',
 '문어',
 '발식',
 '확장',
 '해서',
 '신한',
 '카드',
 '슈퍼',
 '솔',
 '솔',
 '뱅크',
 '다',
 '깔',
 '게',
 '만드',
 '는',
 '거',
 '야',
 '화딱지',
 '남']

단어 인덱스 생성

In [15]:
token= Tokenizer(lower=False)
token.fit_on_texts(tokenized_docs)
print(len(token.word_index))

13010


문장 벡터화

In [16]:
x=token.texts_to_sequences(tokenized_docs)
print(x[0])

[5400, 1682, 170, 11, 159, 37, 2292, 14, 301, 184, 28, 45, 19, 106, 123, 1, 76, 15, 216, 5, 45, 2, 10, 64]


가장 긴 문장의 길이 구하기

In [17]:
max_length=max([len(i) for i in x]) + 1
max_length

302

가장 긴 길이에 맞춰서 패딩
* RNN의 경우 패딩을 post로 주는 것이 더 좋음. 단어가 앞, 0이 뒤에 붙는
* Transformer 계열(gpt)은 위치 정보가 따로 있으므로 post, pre든 차이가 없음

In [18]:
X_padded=pad_sequences(x, maxlen=max_length, padding='post')
print(X_padded[0])

[5400 1682  170   11  159   37 2292   14  301  184   28   45   19  106
  123    1   76   15  216    5   45    2   10   64    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0 

In [19]:
X_padded.shape

(22246, 302)

In [20]:
y=train_data['is_good']
y.value_counts()

is_good
1    13240
0     9006
Name: count, dtype: int64

홀드아웃

In [ ]:
#!pip install scikit-learn

In [21]:
from sklearn.model_selection import train_test_split

In [22]:
X_train, X_valid, y_train, y_valid= train_test_split(X_padded, y, test_size=0.3, stratify=y, random_state=42)

In [23]:
# 임베딩에 입력할 단어수 추출
word_size=len(token.word_index)
print(word_size)

13010


In [24]:
import joblib

In [25]:
joblib.dump(token, "./model/bank_app_tokeizer.joblib")
joblib.dump(max_length, "./model/bank_app_max_length.joblib")

['./model/bank_app_max_length.joblib']

# 양방향 RNN 네트워크를 이용해 텍스트 분석

In [26]:
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [27]:
birnn=Sequential()
birnn.add(Input(shape=(max_length, )))
birnn.add(Embedding(input_dim=word_size, output_dim=64))
birnn.add(Bidirectional(SimpleRNN(128, return_sequences=False, activation='tanh')))
birnn.add(Dense(32, activation='relu'))
birnn.add(Dropout(0.5))
birnn.add(Dense(1, activation='sigmoid'))
birnn.summary()

I0000 00:00:1747960062.013678     680 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1347 MB memory:  -> device: 0, name: NVIDIA GeForce MX450, pci bus id: 0000:01:00.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 302, 64)        │       832,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 256)            │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 890,305 (3.40 MB)

 Trainable params: 890,305 (3.40 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
birnn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
early_stop=EarlyStopping(patience=10, restore_best_weights=True)
model_path="./model/bank_app_review_birnn.keras"
checkpoint=ModelCheckpoint(filepath=model_path, monitor='val_loss',
                          save_best_only=True, 
                          verbose=1)
birnn_history=birnn.fit(X_train, y_train, epochs=1000, batch_size=64, validation_data=(X_valid, y_valid),
                       callbacks=[early_stop, checkpoint])


Epoch 1/1000


I0000 00:00:1747960069.064129     729 service.cc:152] XLA service 0x7ff8e80106a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1747960069.064172     729 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce MX450, Compute Capability 7.5
2025-05-23 09:27:49.139996: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1747960069.593480     729 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/244 ━━━━━━━━━━━━━━━━━━━━ 22:29 6s/step - accuracy: 0.4688 - auc: 0.5193 - loss: 0.6882

I0000 00:00:1747960072.177063     729 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


244/244 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - accuracy: 0.7210 - auc: 0.7687 - loss: 0.5399
Epoch 1: val_loss improved from inf to 0.28642, saving model to ./model/bank_app_review_birnn.keras
244/244 ━━━━━━━━━━━━━━━━━━━━ 55s 202ms/step - accuracy: 0.7213 - auc: 0.7692 - loss: 0.5395 - val_accuracy: 0.8840 - val_auc: 0.9484 - val_loss: 0.2864
Epoch 2/1000
244/244 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - accuracy: 0.8787 - auc: 0.9337 - loss: 0.3226
Epoch 2: val_loss did not improve from 0.28642
244/244 ━━━━━━━━━━━━━━━━━━━━ 38s 155ms/step - accuracy: 0.8786 - auc: 0.9336 - loss: 0.3227 - val_accuracy: 0.6596 - val_auc: 0.8647 - val_loss: 0.6403
Epoch 3/1000
244/244 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - accuracy: 0.7811 - auc: 0.8509 - loss: 0.4700
Epoch 3: val_loss did not improve from 0.28642
244/244 ━━━━━━━━━━━━━━━━━━━━ 40s 166ms/step - accuracy: 0.7813 - auc: 0.8512 - loss: 0.4696 - val_accuracy: 0.7382 - val_auc: 0.9055 - val_loss: 0.4939
Epoch 4/1000
244/244 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/

KeyboardInterrupt: 

In [ ]:
joblib.dump(birnn,"./model/birnn.joblib")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(birnn_history.history['loss'])
plt.plot(birnn_history.history['val_loss'])
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend(['train','valid'])

In [ ]:
print(birnn.evaluate(X_valid, y_valid))

In [ ]:
pred=birnn.predict(X_valid)
pred

In [ ]:
y_valid

In [ ]:
pred=pd.DataFrame(pred)

In [ ]:
y_valid = pd.DataFrame(y_valid.reset_index(drop=True))
y_valid

In [ ]:
result = pd.concat([y_valid, pred], axis=1)
result

In [ ]:
result.columns = ['is_good', 'pred']
result

In [ ]:
result['pred'] = result['pred'].apply(lambda x: 1 if x >= 0.5 else 0)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(result['is_good'], result['pred']))

# LSTM과 CNN 조합 모델로 분석

In [32]:
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, GlobalMaxPooling1D


In [33]:
lstm_cnn = Sequential()
lstm_cnn.add(Input(shape=(max_length,)))
lstm_cnn.add(Embedding(input_dim=word_size, output_dim=128))
lstm_cnn.add(Dropout(0.3))
lstm_cnn.add(Conv1D(128, 5, padding='valid', activation='relu'))
lstm_cnn.add(MaxPooling1D(pool_size=4))
lstm_cnn.add(Conv1D(128, 5, padding='valid', activation='relu'))
lstm_cnn.add(MaxPooling1D(pool_size=4))
lstm_cnn.add(Bidirectional(LSTM(256, return_sequences=True)))
lstm_cnn.add(GlobalMaxPooling1D())
lstm_cnn.add(Dense(64, activation='relu'))
lstm_cnn.add(Dropout(0.3))
lstm_cnn.add(Dense(32, activation='relu'))
lstm_cnn.add(Dense(1, activation='sigmoid'))
lstm_cnn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (None, 302, 128)       │     1,665,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 302, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 298, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 74, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 70, 128)        │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 17, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 17, 512)        │       788,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 512)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,652,801 (10.12 MB)

 Trainable params: 2,652,801 (10.12 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_cnn.compile(loss='binary_crossentropy', optimizer='adam',
             metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
early_stop = EarlyStopping(patience=10, restore_best_weights=True)
model_path = "./model/bank_app_review_lstm_cnn.keras"
checkpoint = ModelCheckpoint(filepath=model_path, monitor='val_loss',
                            save_best_only=True,
                            verbose=1)
lstm_cnn_history = lstm_cnn.fit(X_train, y_train, epochs=1000, batch_size=64,
                         validation_data=(X_valid, y_valid),
                         callbacks=[early_stop, checkpoint])
plt.figure(figsize=(8, 5))
plt.plot(lstm_cnn_history.history['loss'])
plt.plot(lstm_cnn_history.history['val_loss'])
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend(['train', 'valid'])
plt.show()

In [ ]:
print(lstm_cnn.evaluate(X_valid, y_valid))

In [ ]:
pred = lstm_cnn.predict(X_valid)
pred

In [ ]:
pred = pd.DataFrame(pred)

In [ ]:
y_valid = pd.DataFrame(y_valid.reset_index(drop=True))
y_valid

In [ ]:
result = pd.concat([y_valid, pred], axis=1)
result

In [ ]:
result.columns = ['is_good', 'pred']
result

In [ ]:
result['pred'] = result['pred'].apply(lambda x: 1 if x > 0.5 else 0)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(result['is_good'], result['pred']))


# Attention=>GPT
* RNN이나 LSTM은 전체 문장을 기억
* 중요한 단어에만 집중

In [ ]:
#!pip install attention

In [34]:
from attention import Attention

In [35]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling1D, LayerNormalization, MultiHeadAttention, Add

In [36]:
inputs = Input(shape=(max_length,))
x = Embedding(input_dim=word_size, output_dim=128)(inputs)
x = Dropout(0.3)(x)
# 양방향 LSTM
x = Bidirectional(LSTM(128, return_sequences=True))(x)
# 멀티헤드 어텐션
attn_output = MultiHeadAttention(num_heads=4, key_dim=64)(x, x)
attn_output = Dropout(0.3)(attn_output)
x = Add()([x, attn_output])
x = LayerNormalization()(x)
x = GlobalAveragePooling1D()(x)
# DNN
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dense(32, activation='relu')(x)
# 출력층
outputs = Dense(1, activation='sigmoid')(x)
attn_model = Model(inputs=inputs, outputs=outputs)
attn_model.summary()

Model: "functional_17"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 302)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 302, 128)  │  1,665,280 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 302, 128)  │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 302, 256)  │    263,168 │ dropout_4[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 302, 256)  │    263,168 │ bidirectional_2[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 302, 256)  │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 302, 256)  │          0 │ bidirectional_2[… │
│                     │                   │            │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 302, 256)  │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │     32,896 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 128)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      8,256 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 32)        │      2,080 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1)         │         33 │ dense_7[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,235,393 (8.53 MB)

 Trainable params: 2,235,393 (8.53 MB)

 Non-trainable params: 0 (0.00 B)

In [37]:
attn_model.compile(loss='binary_crossentropy', optimizer='adam',
             metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
early_stop = EarlyStopping(patience=5, restore_best_weights=True)
model_path = "./model/bank_app_review_attn_model.keras"
checkpoint = ModelCheckpoint(filepath=model_path, monitor='val_loss',
                            save_best_only=True,
                            verbose=1)
attn_model_history = attn_model.fit(X_train, y_train, epochs=2000, batch_size=128,
                         validation_data=(X_valid, y_valid),
                         callbacks=[early_stop, checkpoint])
plt.figure(figsize=(8, 5))
plt.plot(attn_model_history.history['loss'])
plt.plot(attn_model_history.history['val_loss'])
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend(['train', 'valid'])
plt.show()

Epoch 1/2000
122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6040 - auc: 0.5771 - loss: 0.6653
Epoch 1: val_loss improved from inf to 0.47148, saving model to ./model/bank_app_review_attn_model.keras
122/122 ━━━━━━━━━━━━━━━━━━━━ 206s 2s/step - accuracy: 0.6045 - auc: 0.5780 - loss: 0.6649 - val_accuracy: 0.7679 - val_auc: 0.8609 - val_loss: 0.4715
Epoch 2/2000
122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7711 - auc: 0.8309 - loss: 0.4621
Epoch 2: val_loss improved from 0.47148 to 0.38555, saving model to ./model/bank_app_review_attn_model.keras
122/122 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - accuracy: 0.7707 - auc: 0.8307 - loss: 0.4624 - val_accuracy: 0.8220 - val_auc: 0.9001 - val_loss: 0.3856
Epoch 3/2000
122/122 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8419 - auc: 0.8933 - loss: 0.3831
Epoch 3: val_loss improved from 0.38555 to 0.30819, saving model to ./model/bank_app_review_attn_model.keras
122/122 ━━━━━━━━━━━━━━━━━━━━ 213s 2s/step - accuracy: 0.8420 - auc: 0.8934

KeyboardInterrupt: 

In [ ]:
print(attn_model.evaluate(X_valid, y_valid))

In [ ]:
pred = attn_model.predict(X_valid)
pred

In [ ]:
pred = pd.DataFrame(pred)

In [ ]:
y_valid = pd.DataFrame(y_valid.reset_index(drop=True))
y_valid

In [ ]:
result = pd.concat([y_valid, pred], axis=1)
result

In [ ]:

result.columns = ['is_good', 'pred']
result

In [ ]:

result['pred'] = result['pred'].apply(lambda x: 1 if x > 0.5 else 0)

In [ ]:

from sklearn.metrics import classification_report
print(classification_report(result['is_good'], result['pred']))

In [ ]:
joblib.dump(birnn,"./model/birnn.joblib")